# Google Search Scraper — Демо-ноутбук

Интерактивное тестирование поиска и скрапинга.
Результаты сохраняются в папку `results/`.

In [ ]:
import subprocess
import sys
import os
import json
from datetime import datetime
from pathlib import Path

# ============================================================
# НАСТРОЙКА: укажите путь к корню проекта niko_code
# Если ноутбук лежит в notebooks/ внутри проекта — оставьте как есть.
# Если скопировали ноутбук в другое место — укажите путь вручную.
# ============================================================
PROJECT_ROOT = Path(os.getcwd()).parent

# Автопоиск: если рядом нет пакета, ищем по типичным путям
_candidates = [
    PROJECT_ROOT,
    Path(os.getcwd()),
    Path("/home/user/niko_code"),
    Path("/app"),
    Path("/workspace"),
]
for _p in _candidates:
    if (_p / "google_search_scraper" / "__init__.py").exists():
        PROJECT_ROOT = _p
        break

# Установить зависимости если нужно
try:
    import requests  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])

# Добавить корень проекта в sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Папка для результатов
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

from google_search_scraper.search_providers import DuckDuckGoProvider
from google_search_scraper.scraper import PageScraper
from google_search_scraper.core import SearchAndScrape

print(f"Проект: {PROJECT_ROOT}")
print(f"Результаты: {RESULTS_DIR}")
print("Готово!")

## 1. Тест поиска DuckDuckGo

In [ ]:
provider = DuckDuckGoProvider()
results = provider.search("Python web scraping tutorial", num_results=5)

print(f"Найдено результатов: {len(results)}\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r.title}")
    print(f"   URL: {r.url}")
    print(f"   {r.snippet[:100]}...\n")

## 2. Тест скрапинга страниц

In [ ]:
scraper = PageScraper(timeout=20, max_workers=3)

# Скрапим URL-ы из результатов поиска
urls = [r.url for r in results[:3]]
pages = scraper.scrape_urls(urls)

for page in pages:
    status = "OK" if page.success else f"FAIL: {page.error}"
    print(f"[{status}] {page.title}")
    print(f"  URL: {page.url}")
    print(f"  Слов: {page.word_count}")
    if page.success:
        preview = page.text[:200].replace('\n', ' ')
        print(f"  Текст: {preview}...")
    print()

## 3. Полный цикл — поиск + скрапинг + сохранение

In [ ]:
# Настройки
QUERY = "Python requests library tutorial"
NUM_RESULTS = 5

# Запуск
engine = SearchAndScrape(
    provider=DuckDuckGoProvider(),
    scraper=PageScraper(timeout=20, max_workers=3),
    num_results=NUM_RESULTS,
)

# Имя файла с датой и запросом
safe_query = "".join(c if c.isalnum() or c in " -_" else "" for c in QUERY)
safe_query = safe_query.strip().replace(" ", "_")[:40]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = RESULTS_DIR / f"{timestamp}_{safe_query}.json"

# Выполняем и сохраняем
saved_path = engine.run_and_save(QUERY, output_path=str(output_file))
print(f"\nФайл сохранён: {saved_path}")

## 4. Просмотр сохранённых результатов

In [ ]:
with open(output_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Запрос: {data['query']}")
print(f"Время: {data['timestamp']}")
print(f"Найдено: {data['total_results']}")
print(f"Скачано OK: {data['scraped_ok']}")
print(f"Ошибок: {data['scraped_fail']}")
print(f"\n{'='*60}\n")

for i, r in enumerate(data["results"], 1):
    status = "OK" if r["success"] else "FAIL"
    print(f"{i}. [{status}] {r['search_title']}")
    print(f"   URL: {r['url']}")
    print(f"   Слов: {r['word_count']}")
    if r["success"] and r["text"]:
        preview = r["text"][:150].replace('\n', ' ')
        print(f"   Текст: {preview}...")
    elif r["error"]:
        print(f"   Ошибка: {r['error']}")
    print()

## 5. Все сохранённые файлы в results/

In [ ]:
files = sorted(RESULTS_DIR.glob("*.json"))
print(f"Файлов в results/: {len(files)}\n")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")